# Trino and Presto


*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Trino and how is it architected?

Trino is a distributed MPP SQL engine for interactive analytics, forked from PrestoSQL, which came out of the Facebook Presto project. Trino stores no data of its own: it reads from connectors, plans a distributed query, and streams results back.

+ **Coordinator** - parses SQL, resolves names against catalogs, builds and optimizes the plan, splits work into stages, schedules tasks, and tracks query state. Exactly one per cluster.
+ **Workers** - execute tasks, read splits from connectors, exchange intermediate rows, and return results. Scale horizontally.
+ **Discovery service** - workers register their HTTP URLs so the coordinator can schedule onto them. Embedded in the coordinator by default.
+ **Connectors** - per-catalog plugins that expose metadata and read/write APIs over underlying systems.

```text
config.properties (coordinator)
coordinator=true
node-scheduler.include-coordinator=false
http-server.http.port=8080
discovery.uri=http://trino-coordinator:8080
query.max-memory=100GB
query.max-memory-per-node=16GB
```

```text
config.properties (worker)
coordinator=false
http-server.http.port=8080
discovery.uri=http://trino-coordinator:8080
```

A query flows as client -> coordinator (plan) -> stages -> tasks -> splits -> workers -> exchange -> client. Trino pipelines data in memory and streams results, which is why first rows return quickly on interactive queries.

Worked example. A query reads a 1 TB Iceberg table stored as 8,000 files on a 20-worker cluster. The coordinator renames tables, optimizes the plan, and splits it into 3 stages: scan, partial aggregate, final aggregate. The scan stage produces 8,000 splits. Each task takes `task_concurrency` splits at a time (16 by default), so the stage creates about 500 tasks, and the coordinator schedules those tasks across the 20 workers. Data rows move worker to worker through the exchange, never through the coordinator.

Common mistake: treating the coordinator as a data node. Setting `node-scheduler.include-coordinator=true` lets the coordinator execute splits as well, so its heap competes with planning, the Web UI, and query state. On a small test cluster that is a convenience; in production the coordinator should schedule and the workers should read.

In [ ]:
# Splits become tasks; tasks are scheduled onto workers.
splits = 8000
task_concurrency = 16
workers = 20

tasks = -(-splits // task_concurrency)          # ceil division
tasks_per_worker = -(-tasks // workers)
print(f"splits={splits} task_concurrency={task_concurrency}")
print(f"tasks={tasks} tasks_per_worker={tasks_per_worker}")

assert tasks == 500
assert tasks_per_worker == 25

## How do catalogs and connectors work in Trino?

A catalog is a named connection to a data source, backed by exactly one connector. Queries use three-part names: `catalog.schema.table`. Catalogs are files in `etc/catalog/`, and the file name becomes the catalog name.

```text
etc/catalog/iceberg.properties   ->  iceberg.analytics.orders
etc/catalog/postgres.properties  ->  postgres.public.customers
etc/catalog/kafka.properties     ->  kafka.acme.events
```

```sql
SHOW CATALOGS;
SHOW SCHEMAS FROM iceberg;
SHOW TABLES FROM iceberg.analytics;
DESCRIBE iceberg.analytics.orders;

SELECT * FROM system.metadata.catalogs;
```

Practical rules: use one catalog per physical source rather than one per team; assume a connector may be read-only (Kafka) or fully writable (Iceberg); and remember that connectors differ in pushdown capability, so the same SQL can cost very different amounts per catalog.

How resolution works. The coordinator loads every file in `etc/catalog/` at startup and names each catalog after its file. When it sees `iceberg.analytics.orders`, it hands the schema and table to the Iceberg connector, which reads the table metadata from its catalog (REST, Hive metastore, or Glue). The connector is also the only component that knows which predicates it can push down and how it generates splits. You can inspect what the engine knows about catalogs, schemas, and tables without touching the underlying system.

```sql
SELECT table_catalog, table_schema, table_name, table_type
FROM system.jdbc.tables
WHERE table_schema = 'analytics'
ORDER BY table_catalog, table_name;
```

Common mistake: creating one catalog per team or per dashboard. Catalogs are connections, not namespaces. Use one catalog per physical source, one schema per domain, and separate access with rules and grants rather than duplicated connections.

## How do you configure the lakehouse connectors?

The Iceberg, Delta Lake, and Hive connectors cover most lakehouse data. All three use `.properties` files, and the same catalog can be described declaratively in a Helm values file.

```text
etc/catalog/iceberg.properties
connector.name=iceberg
iceberg.catalog.type=rest
iceberg.rest-catalog.uri=https://catalog.acme.internal
iceberg.rest-catalog.warehouse=s3://acme-warehouse/
fs.native-s3.enabled=true
s3.region=us-east-1
```

```text
etc/catalog/delta.properties
connector.name=delta_lake
hive.metastore.uri=thrift://hive-metastore:9083
fs.native-s3.enabled=true
s3.region=us-east-1
```

```text
etc/catalog/hive.properties
connector.name=hive
hive.metastore.uri=thrift://hive-metastore:9083
hive.parquet.use-column-names=true
hive.orc.use-column-names=true
hive.max-split-size=128MB
```

```yaml
# values.yaml for the Trino Helm chart: catalogs declared as YAML
additionalCatalogs:
  iceberg: |
    connector.name=iceberg
    iceberg.catalog.type=hive_metastore
    hive.metastore.uri=thrift://hive-metastore:9083
    fs.native-s3.enabled=true
```

```sql
CREATE TABLE iceberg.analytics.orders (
  order_id     bigint,
  customer_id  bigint,
  order_ts     timestamp(6),
  status       varchar,
  amount       decimal(18, 2)
)
WITH (
  format = 'PARQUET',
  partitioning = ARRAY['day(order_ts)', 'bucket(customer_id, 16)']
);

-- Register an existing Delta table without moving data
CALL delta.system.register_table(schema_name => 'lake', table_name => 'orders_events',
     table_location => 's3://acme-lake/delta/orders_events/');
```

`hive.parquet.use-column-names=true` maps columns by name rather than position, which matters when upstream writers append columns. Prefer `iceberg.catalog.type=rest` for multi-engine interoperability, and keep credentials out of properties files by using environment substitution or an instance role.

How each connector plans. The Iceberg connector reads table metadata from its catalog and uses manifests and column statistics for pruning, so a filter on a partition column skips files before any split is created. The Delta Lake connector reads `_delta_log` in the table directory and uses checkpoint and partition metadata the same way. The Hive connector leans on the metastore for partitions and on the file formats for row-group or stripe pruning, and it depends on a well-maintained metastore for good plans. All three expose the same three-part namespace, so the same SQL runs on all three.

```sql
SHOW CREATE TABLE iceberg.analytics.orders;

SELECT snapshot_id, committed_at, operation
FROM iceberg.analytics."orders$snapshots"
ORDER BY committed_at DESC
LIMIT 5;

SELECT count(*) AS files, sum(file_size_in_bytes) / 1024 / 1024 AS mb
FROM iceberg.analytics."orders$files";
```

Common mistake: pointing the Hive connector at a directory that is really an Iceberg or Delta table. The query still returns rows, but you lose snapshot semantics, hidden partitioning, and metadata pruning, and the plan degrades to a file listing. Match the connector to the table format.

## How do you query operational databases and warehouses?

The PostgreSQL and BigQuery connectors are the standard federation path for joining lake data with live sources.

```text
etc/catalog/postgres.properties
connector.name=postgresql
connection-url=jdbc:postgresql://pg-primary.acme.internal:5432/app
connection-user=trino_reader
connection-password=${ENV:PG_TRINO_PASSWORD}
```

```text
etc/catalog/bigquery.properties
connector.name=bigquery
bigquery.project-id=acme-prod
bigquery.credentials-file=/etc/trino/gcp-credentials.json
bigquery.max-parallelism=8
```

```sql
SELECT o.order_id, c.tier, b.lifetime_value
FROM iceberg.analytics.orders o
JOIN postgres.public.customers c ON o.customer_id = c.customer_id
LEFT JOIN bigquery.crm.customer_value b ON b.customer_id = c.customer_id
WHERE o.order_date = DATE '2026-10-04'
  AND c.live = true;
```

Keep the JDBC-side filter selective: Trino pushes predicates into the generated SQL, but a scan of a 500 million row Postgres table through a handful of splits is still slow. The BigQuery connector reads through the BigQuery API, so bytes billed still accrue on the BigQuery side; use it for occasional federation, not as the primary compute for a petabyte warehouse.

How JDBC reads work. The PostgreSQL connector translates the pushed-down part of the predicate into the SQL it sends to the source, then generates splits over the matching row ranges. Trino does the joins, aggregations, and final filter itself. A remote scan is therefore only as fast as the source: indexes matter, a full table scan on the source costs the same as it would in the source, and the connector cannot push a Trino function that has no SQL equivalent.

```sql
EXPLAIN (TYPE DISTRIBUTED)
SELECT count(*) FROM postgres.public.customers
WHERE tier = 'gold' AND live = true;

SELECT query_id, state, query
FROM system.runtime.queries
WHERE state = 'RUNNING'
ORDER BY created DESC
LIMIT 10;
```

Common mistake: assuming the whole query is pushed into the source. Only the scan predicate and projection go down; `JOIN`, `GROUP BY`, and window functions run in Trino unless the connector explicitly supports them. Keep the pushed predicate selective so the remote scan stays small.

## How do you query Kafka from Trino?

The Kafka connector exposes a topic as a table for ad-hoc inspection. It is read-only.

```text
etc/catalog/kafka.properties
connector.name=kafka
kafka.nodes=kafka-1:9092,kafka-2:9092
kafka.table-names=acme.orders
kafka.table-description-dir=/etc/trino/kafka
```

```json
{
  "tableName": "acme.orders",
  "schemaName": "acme",
  "topicName": "orders",
  "message": {
    "dataFormat": "json",
    "fields": [
      { "name": "order_id", "type": "BIGINT", "mapping": "order_id" },
      { "name": "customer_id", "type": "BIGINT", "mapping": "customer_id" },
      { "name": "amount", "type": "DOUBLE", "mapping": "amount" }
    ]
  }
}
```

```sql
SELECT _partition_id, _partition_offset, order_id, amount
FROM kafka.acme."orders"
WHERE _timestamp > now() - INTERVAL '15' MINUTE
LIMIT 100;
```

Internal columns (`_partition_id`, `_partition_offset`, `_message`, `_timestamp`) make this useful for debugging a producer. It is not a streaming engine: each query re-reads the log from the configured offset, so use Flink, Spark Structured Streaming, or ClickHouse for continuous processing.

How the scan works. Each Kafka partition becomes at least one split, so the number of splits tracks topic parallelism rather than message count. The connector maps JSON or Avro fields to columns using the table description, and it exposes internal columns for partition, offset, message, and timestamp. There is no predicate pushdown: a `WHERE` clause is evaluated by Trino after decoding, so a filter does not reduce how much of the log is read.

```sql
SELECT _partition_id,
       max(_partition_offset) AS last_offset,
       count(*) AS messages
FROM kafka.acme."orders"
WHERE _timestamp > now() - INTERVAL '1' HOUR
GROUP BY _partition_id
ORDER BY _partition_id;
```

Common mistake: expecting a table to appear for every topic automatically. The topic must be listed in `kafka.table-names` or discovered through `kafka.table-description-dir`, and the connector is read-only. Use it for inspection and debugging, not for continuous processing.

## How do you run federated queries across catalogs?

Federation is the headline Trino feature: one SQL statement joins sources in different systems. The coordinator plans one query, but data must physically move to where the join happens.

```sql
SELECT p.region, count(*) AS orders, sum(p.amount) AS revenue
FROM postgres.public.payments p
JOIN iceberg.analytics.orders o ON p.order_id = o.order_id
WHERE o.order_date BETWEEN DATE '2026-10-01' AND DATE '2026-10-04'
GROUP BY p.region;
```

```sql
SELECT 'iceberg' AS src, count(*) FROM iceberg.analytics.orders
UNION ALL
SELECT 'delta', count(*) FROM delta.lake.orders
UNION ALL
SELECT 'hive', count(*) FROM hive.warehouse.orders;
```

Rules that keep federation fast:

+ Push selective predicates into each source; the connector sees only the filters it supports.
+ Broadcast small dimension tables instead of shuffling the fact side.
+ Materialize a repeatedly joined external table into the lake when it is large.
+ Remember that a cross-catalog join cannot use one catalog's native acceleration for the whole plan.

What physically happens. The coordinator plans one query, and each catalog contributes a scan fragment. Because the rows live in different systems, at least one side of every cross-catalog join must move over the network through an exchange. The optimizer decides which side moves and how, using the statistics each connector reports. If one connector reports no statistics, the plan falls back to heuristics and can move far more data than necessary.

```sql
EXPLAIN (TYPE DISTRIBUTED)
SELECT p.region, count(*) AS orders
FROM postgres.public.payments p
JOIN iceberg.analytics.orders o ON p.order_id = o.order_id
WHERE o.order_date = DATE '2026-10-04'
GROUP BY p.region;
```

Common mistake: joining two large external sources and expecting each one to be filtered remotely. With no selective predicate on either side, both scans are full scans and the exchange is large. Add a selective predicate, materialize the smaller side into the lake, or move the join to the system that owns most of the data.

## How does query planning work and how do you read EXPLAIN?

Trino turns SQL into a logical plan, optimizes it, then creates an executable distributed plan of fragments and stages. `EXPLAIN` exposes each layer.

```sql
EXPLAIN (TYPE LOGICAL)
SELECT customer_id, sum(amount) FROM iceberg.analytics.orders GROUP BY customer_id;

EXPLAIN (TYPE DISTRIBUTED)
SELECT customer_id, sum(amount) FROM iceberg.analytics.orders GROUP BY customer_id;

EXPLAIN (TYPE IO)
SELECT customer_id, sum(amount) FROM iceberg.analytics.orders
WHERE order_date = DATE '2026-10-04' GROUP BY customer_id;

EXPLAIN ANALYZE
SELECT customer_id, sum(amount) FROM iceberg.analytics.orders GROUP BY customer_id;
```

What to look for:

+ `TableScan`: `constraint`, `layout`, and `dynamicFilters` show what was pushed down.
+ `Exchange` with `type = REPARTITION`: data is being shuffled between stages.
+ `type = GATHER`: results stream to the coordinator. Fine for final output, dangerous mid-plan.
+ `Join`: `distribution = PARTITIONED` or `BROADCAST`, and which side is the build side.
+ `EXPLAIN ANALYZE` gives per-operator row counts and wall time, so an estimate that is orders of magnitude off tells you statistics are missing.

How a distributed plan is organized. A query is one or more stages connected by exchanges. A fragment prints per stage; a task is one instance of a fragment on one worker, and a split is one unit of input inside a task. `EXPLAIN (TYPE DISTRIBUTED)` prints the fragment graph with `RemoteSource` nodes where one stage feeds another, so you can read the plan as a tree from the final output back to the scans.

```text
Fragment 0 [HASH]
    Output layout: [customer_id, sum]
    Aggregate(FINAL)
    └─ RemoteSource[1]
Fragment 1 [HASH]
    Aggregate(PARTIAL)
    └─ RemoteSource[2]
Fragment 2 [SOURCE]
    TableScan[table = iceberg:analytics.orders]
        Layout: [customer_id, amount]
        Estimates: {rows: 50000000 (2.0GB)}
```

Read it bottom-up. Fragment 2 scans, fragment 1 pre-aggregates locally, fragment 0 merges the partial results. An `Exchange` with `type = REPARTITION` is a shuffle; `type = GATHER` sends rows to the coordinator and is only safe for small final results. Common mistake: reading the `Estimates` as measured facts. They are the optimizer's guesses, and a guess that is orders of magnitude off is the signal that statistics are missing.

## How do predicate and projection pushdown work?

Projection pushdown means only referenced columns are read. Predicate pushdown means filters are evaluated in the source, so fewer rows cross the network.

```sql
EXPLAIN (TYPE IO)
SELECT order_id, amount
FROM iceberg.analytics.orders
WHERE order_date = DATE '2026-10-04'
  AND status = 'shipped'
  AND customer_id IN (101, 102, 103);
```

The IO plan reports a `domain` per column; an unbounded domain means the connector cannot help and Trino filters after reading.

```text
TableScan[table = iceberg:analytics.orders]
    Layout: [order_id, amount]
    Constraint: order_date IN [2026-10-04], customer_id IN [101, 102, 103], status IN [shipped]
```

Practical levers:

+ Filter on partition columns so file pruning happens before any split is created.
+ Avoid wrapping filter columns in functions. `date_trunc('day', ts) = ...` blocks pruning; use a range instead.
+ Cast literals to the column type so the connector can compare them directly.
+ Check the connector docs for the functions and predicate shapes it can push down.

How the negotiation works. The optimizer computes a `Domain` per column, which is a set of allowed values or ranges. The connector accepts the domains it can translate and ignores the rest. Accepted predicates are applied while the split is read, and Trino re-checks them afterwards, so a missed pushdown costs time but never changes the answer. Projection pushdown is separate: the connector reads only the referenced columns from columnar formats.

```sql
EXPLAIN (TYPE IO)
SELECT order_id, amount
FROM iceberg.analytics.orders
WHERE order_date = DATE '2026-10-04'
  AND amount > 100.00;
```

The IO plan lists the domain per column. Here `order_date` is a partition column, so the domain prunes whole files, and `amount` prunes row groups through column statistics. Common mistake: wrapping the column in a function, as in `date_trunc('day', order_ts) = DATE '2026-10-04'`. That produces an unbounded domain, so the connector reads everything and Trino filters afterwards. Rewrite it as a range on the bare column.

## How do broadcast partitioned and dynamic-filter joins work?

Trino builds a hash table from one side of a join and probes it with the other. Choosing the distribution decides how much data moves.

+ **Broadcast**: the build side is copied to every worker. Cheap when small, fatal when not.
+ **Partitioned**: both sides are repartitioned by join key so each worker sees matching keys. Needed for large-to-large joins.
+ **AUTOMATIC** (default): the optimizer decides from statistics and falls back to broadcast when the build side looks small.
+ **Dynamic filtering**: at runtime Trino collects min/max, distinct values, and bloom filters from the build side and applies them to the probe-side scan before splits are scheduled.

```sql
SET SESSION join_distribution_type = 'BROADCAST';
SET SESSION join_max_broadcast_table_size = '64MB';
SET SESSION enable_dynamic_filtering = true;
SET SESSION dynamic_filtering_wait_timeout = '20s';

EXPLAIN (TYPE DISTRIBUTED)
SELECT f.order_id, d.region
FROM iceberg.analytics.fact_orders f
JOIN iceberg.analytics.dim_order d ON f.order_id = d.order_id
WHERE d.region = 'EMEA';
```

```text
TableScan[table = iceberg:analytics.fact_orders]
    Dynamic filters: order_id BETWEEN 1000000 AND 1002000
```

Warning signs: a broadcast node after an inexplicably large aggregation, or raising `join_max_broadcast_table_size` globally to hide a bad estimate. The right fix is usually correct statistics, not a bigger broadcast limit. The dynamic-filtering wait timeout is the trade-off: a longer wait lets the filter arrive before scheduling and saves IO, but adds latency when the build side is slow.

How the runtime filter is built. The build side is scanned first and a hash table is built. At the same time Trino collects the join keys from the build side and attaches a dynamic filter to the probe-side scan. In a broadcast join the filter is applied inside the local scan on each worker; in a partitioned join the coordinator gathers the values and pushes them to the probe-side scans before all splits are scheduled. `EXPLAIN` shows this as `dynamicFilterAssignments` on the join and `dynamicFilters` on the scan.

```sql
EXPLAIN (TYPE DISTRIBUTED)
SELECT f.order_id, d.region
FROM iceberg.analytics.fact_orders f
JOIN iceberg.analytics.dim_order d ON f.order_id = d.order_id
WHERE d.region = 'EMEA';
```

```text
InnerJoin[...]
    dynamicFilterAssignments = {order_id -> #df_370}
└─ TableScan[table = iceberg:analytics.fact_orders]
       dynamicFilters = {order_id = #df_370}
```

Common mistake: forcing `BROADCAST` to make a slow join faster. A broadcast copies the build side to every worker, so a build side that is larger than the estimate multiplies network and memory by the worker count. Fix the statistics and let `AUTOMATIC` choose, or set `PARTITIONED` explicitly for large-to-large joins.

## How does the cost-based optimizer use statistics and ANALYZE?

The CBO estimates cardinality and cost from table and column statistics: row count, distinct values, null fraction, and data size. Those estimates drive join ordering, join distribution, and aggregation strategy.

```sql
SET SESSION optimizer.join_reordering_strategy = 'AUTOMATIC';

SHOW STATS FOR iceberg.analytics.orders;

ANALYZE iceberg.analytics.orders;
ANALYZE iceberg.analytics.orders WITH (columns = ARRAY['order_date', 'status']);
```

```text
column_name | data_size | distinct_values_count | nulls_fraction | row_count | low_value | high_value
------------+-----------+-----------------------+----------------+-----------+-----------+-----------
order_id    | 4.0E8     | 5.0E7                 | 0.0            | 5.0E7     | 1         | 50000000
status      | 1.5E8     | 8.0                   | 0.0            | NULL      | NULL      | NULL
```

If `row_count` is missing, Trino falls back to heuristics and may broadcast something enormous. Statistics are per-connector: Iceberg reads them from manifests, Hive from the metastore, and JDBC connectors estimate from the source catalog. Re-run `ANALYZE` after bulk loads or major compactions, not on every small append, and prefer connectors with native statistics collection over JDBC sources where `ANALYZE` triggers an expensive scan.

How estimates flow. Connectors report table statistics (row count, data size) and column statistics (distinct values, null fraction, low and high value). The CBO multiplies them through filters, joins, and aggregations to estimate cardinality and cost. Those estimates decide the join order under `optimizer.join-reordering-strategy=AUTOMATIC`, the join distribution under `join_distribution_type=AUTOMATIC`, and the hash partition count. When a statistic is missing, the estimate degrades and every later decision inherits the error.

```sql
SHOW STATS FOR iceberg.analytics.orders;

ANALYZE iceberg.analytics.orders WITH (columns = ARRAY['order_date', 'status']);

SET SESSION join_reordering_strategy = 'AUTOMATIC';
SET SESSION max_reordered_joins = 8;
```

Common mistake: running `ANALYZE` after every small append. It triggers a scan and it is only worth it after bulk loads, schema changes, or major compactions. The opposite mistake is worse: never analyzing, so the CBO keeps broadcasting a table it thinks is small.

In [ ]:
# Will the CBO pick broadcast? Compare build-side bytes with the limit.
build_rows = 20_000
row_bytes = 60
join_max_broadcast_table_size = 100 * 1024 ** 2   # default 100 MB

build_bytes = build_rows * row_bytes
print(f"build side = {build_bytes / 1024 ** 2:.1f} MiB")
print(f"limit      = {join_max_broadcast_table_size / 1024 ** 2:.0f} MiB")

assert build_bytes < join_max_broadcast_table_size

## How does spill to disk work?

Trino normally keeps intermediate state in memory. Spilling lets memory-intensive operators write to local disk instead of failing the query. It is a safety valve, not a performance feature.

```text
etc/config.properties
spill-enabled=true
spiller-spill-path=/mnt/trino-spill
max-spill-per-node=400GB
query-max-spill-per-node=100GB
spill-compression-codec=ZSTD
```

```sql
SET SESSION spill_enabled = true;   -- covers joins, aggregations, ORDER BY, and window functions

SELECT customer_id, count(*) AS orders
FROM iceberg.analytics.orders
GROUP BY customer_id
ORDER BY orders DESC;
```

Supported operators include hash joins, aggregations, sorts, and window functions. Size the spill path for local NVMe, never network storage, and monitor `spill_used_bytes` so spilling does not silently become the normal path.

What actually spills. When memory pressure arrives, the memory manager revokes memory from an operator. A hash join partitions its build table `task.concurrency` ways, spills a subset of those partitions together with the matching probe rows, and reads them back one at a time. Peak memory therefore falls toward the size of the largest partition, roughly the whole build table divided by `task.concurrency` when the keys are even. Aggregations, sorts, and window functions spill intermediate results in the same way.

```text
etc/config.properties
spill-enabled=true
spiller-spill-path=/mnt/nvme0/trino-spill,/mnt/nvme1/trino-spill
spiller-threads=8
spiller-max-used-space-threshold=0.9
max-spill-per-node=400GB
query-max-spill-per-node=100GB
spill-compression-codec=ZSTD
```

Common mistake: pointing the spill path at the JVM log drive or at network storage. Spill is IO heavy, and a saturated system drive can pause the JVM and fail queries. Also treat spill as a safety valve rather than a speedup: a spilled query can be an order of magnitude slower, and pure spill-to-disk is now legacy in favor of fault-tolerant execution.

## How do you manage memory and resource groups?

Two layers control memory: per-query node limits in `config.properties`, and admission control through resource groups.

```text
etc/config.properties
query.max-memory=100GB
query.max-memory-per-node=8GB
query.max-total-memory-per-node=12GB
memory.heap-headroom-per-node=25%
```

```text
etc/resource-groups.properties
resource-groups.configuration-manager=file
resource-groups.config-file=etc/resource-groups.json
```

```json
{
  "rootGroups": [
    {
      "name": "global",
      "softMemoryLimit": "80%",
      "hardConcurrencyLimit": 100,
      "maxQueued": 1000,
      "subGroups": [
        { "name": "etl", "softMemoryLimit": "60%", "hardConcurrencyLimit": 20, "maxQueued": 200, "schedulingPolicy": "fair" },
        { "name": "adhoc", "softMemoryLimit": "30%", "hardConcurrencyLimit": 60, "maxQueued": 800, "schedulingPolicy": "fair" }
      ]
    }
  ],
  "selectors": [
    { "source": ".*airflow.*", "group": "global.etl" },
    { "user": "bi_service", "group": "global.adhoc" },
    { "group": "global.adhoc" }
  ]
}
```

The pattern that works in production: give ETL a large memory share but modest concurrency, give ad-hoc large concurrency but a smaller memory share, and cap queued queries so clients fail fast instead of timing out invisibly.

How admission control works. A query belongs to exactly one resource group and consumes the limits of that group and its ancestors. Running out of memory or concurrency does not kill running queries; new queries queue. `softMemoryLimit` is the point where new queries start to queue, `hardConcurrencyLimit` is the maximum running at once, and `maxQueued` is where the queue rejects new arrivals. `schedulingPolicy` decides which queued query starts next: `fair`, `weighted_fair`, `weighted`, or `query_priority`.

```json
{
  "rootGroups": [
    {
      "name": "global",
      "softMemoryLimit": "80%",
      "hardConcurrencyLimit": 60,
      "maxQueued": 200,
      "jmxExport": true,
      "subGroups": [
        { "name": "etl", "softMemoryLimit": "60%", "hardConcurrencyLimit": 10,
          "maxQueued": 100, "schedulingWeight": 300, "schedulingPolicy": "weighted_fair" },
        { "name": "adhoc", "softMemoryLimit": "30%", "hardConcurrencyLimit": 40,
          "maxQueued": 800, "schedulingWeight": 100, "schedulingPolicy": "weighted_fair" }
      ]
    }
  ],
  "selectors": [
    { "source": ".*airflow.*", "group": "global.etl" },
    { "user": "bi_service", "group": "global.adhoc" },
    { "group": "global.adhoc" }
  ]
}
```

```sql
SELECT resource_group_id, count(*) AS running
FROM system.runtime.queries
WHERE state = 'RUNNING'
GROUP BY resource_group_id
ORDER BY running DESC;
```

Common mistake: setting `maxQueued` to zero or a tiny number and then blaming the engine for rejecting queries. A small queue is intentional back pressure. Size it so clients fail fast under overload instead of waiting until they time out.

## How does Trino handle fault tolerance and retries?

Trino fails a query by default when a worker dies mid-query. Two mechanisms improve resilience: task-level retries and fault-tolerant execution with a durable exchange.

```text
etc/config.properties
retry-policy=TASK
query.max-run-time=6h
query.max-execution-time=5h
```

```text
etc/exchange-manager.properties
exchange-manager.name=filesystem
exchange.base-directories=s3://acme-trino-exchange/
exchange.s3.region=us-east-1
```

How the policies differ:

+ `NONE` - no automatic retry; any worker failure fails the query.
+ `TASK` - failed tasks retry on another worker. Works with the default in-memory exchange for transient errors.
+ `QUERY` - the whole query restarts, which is simpler but more expensive.
+ Fault-tolerant execution (`retry-policy=TASK` plus an exchange manager) persists intermediate data so long ETL queries survive worker restarts.

Set `query.max-run-time` deliberately: without it, a runaway query can occupy a resource group slot for hours.

What fault-tolerant execution adds. With `retry-policy=TASK`, each task is retried on another worker after a failure, and intermediate exchange data is spooled to durable storage so the retry can reuse it. That requires an exchange manager; without one, spooling is limited to the coordinator's in-memory buffer (`exchange.deduplication-buffer-size`, 32 MB by default) and large queries fail with a message about the missing exchange manager. Spooled data is encrypted with a per-query key.

```text
etc/config.properties
retry-policy=TASK
task-retry-attempts-per-task=4
retry-initial-delay=10s
retry-max-delay=1m
query.max-run-time=6h

etc/exchange-manager.properties
exchange-manager.name=filesystem
exchange.base-directories=s3://acme-trino-exchange/
exchange.s3.region=us-east-1
```

Common mistake: turning on `retry-policy=TASK` without an exchange manager, or on a connector that does not support retries. Trino fails such queries with a connector-support error. Also group long batch queries on a cluster with `TASK` retries and keep short interactive queries on a separate cluster with `QUERY` retries or none, because task-level spooling adds latency to small queries.

## Which session properties are most useful for tuning?

Session properties override coordinator defaults per query and per session. They are the first lever before touching cluster config.

```sql
SET SESSION join_distribution_type = 'PARTITIONED';
SET SESSION join_max_broadcast_table_size = '64MB';
SET SESSION enable_dynamic_filtering = true;
SET SESSION task_concurrency = 16;
SET SESSION task_max_writer_count = 8;
SET SESSION max_hash_partition_count = 256;
SET SESSION distinct_aggregations_strategy = 'MARK_DISTINCT';

SHOW SESSION;
RESET SESSION join_distribution_type;
```

| Property | Effect | When to change it |
|---|---|---|
| `join_distribution_type` | broadcast vs partitioned joins | large-to-large joins misplanned |
| `join_max_broadcast_table_size` | build-side size cutoff | broadcast OOM or spill |
| `enable_dynamic_filtering` | runtime probe-side pruning | selective joins on huge fact tables |
| `task_concurrency` | concurrent splits per task | small splits or high-latency storage |
| `task_max_writer_count` | writer parallelism per task | too many small output files |
| `max_hash_partition_count` | shuffle partitions for hash operations | skew or tiny partitions |
| `distinct_aggregations_strategy` | strategy for `DISTINCT` aggregations | grouped distinct queries |
| `query_max_run_time` | per-query timeout | protect shared clusters |

Apply them where they are reproducible: in the CLI or BI connection for interactive work, and in the client session for scheduled jobs. Do not rely on one engineer's shell history.

How precedence works. A configuration property is the cluster default, a session property overrides it for one session, and a system session property changes it for every query while the cluster is running. `SET SESSION` applies to the current session and `RESET SESSION` restores the default. Because clients open their own sessions, anything set in one engineer's interactive shell does not reach a scheduled job unless the job sets it.

```sql
SET SESSION join_distribution_type = 'PARTITIONED';
SET SESSION task_concurrency = 16;
SHOW SESSION LIKE 'join%';
RESET SESSION join_distribution_type;
```

Common mistake: copying a session property out of a tuning blog and applying it globally. A property that helps one skewed query can slow every other query in the cluster. Bind overrides to a client source or a resource group instead of editing the cluster default, and record why each override exists.

## How do you secure Trino with authentication access control and masking?

Security has three independent layers: transport, authentication, and authorization.

```text
etc/config.properties
http-server.https.enabled=true
http-server.https.port=8443
http-server.https.keystore.path=/etc/trino/keystore.jks
http-server.authentication.type=PASSWORD
```

```text
etc/password-authenticator.properties
password-authenticator.name=ldap
ldap.url=ldaps://ldap.acme.internal:636
ldap.user-bind-pattern=uid=${USER},ou=people,dc=acme,dc=internal
```

OAuth2 and JWT are alternatives (`http-server.authentication.type=oauth2` with `http-server.authentication.oauth2.issuer`), and `sql-standard` access control lets Trino enforce `GRANT` and `REVOKE` when catalogs already have a permission model. File-based rules are the most common custom policy.

```text
etc/access-control.properties
access-control.name=file
security.config-file=etc/rules.json
```

```json
{
  "catalogs": [
    { "user": "etl_user", "catalog": "iceberg", "allow": "all" },
    { "user": "analyst", "catalog": "iceberg", "allow": "read-only" }
  ],
  "tables": [
    {
      "user": "analyst",
      "catalog": "iceberg",
      "schema": "analytics",
      "table": "customers",
      "privileges": ["SELECT"],
      "columns": [
        { "name": "email", "mask": "regexp_replace(email, '(^.)[^@]*', '$1***')" },
        { "name": "national_id", "mask": "'REDACTED'" }
      ]
    }
  ]
}
```

```sql
-- The analyst still gets rows, but never the raw values
SELECT customer_id, email, national_id FROM iceberg.analytics.customers LIMIT 5;
```

A table rule can also carry a row filter (`"filter": "region = 'EMEA'"`), which removes rows rather than masking values. Design notes: masks are evaluated per row, so keep expressions simple; masking is presentation control, not encryption, so combine it with storage encryption and least-privilege grants; and test rules with a real non-admin account before rollout, because a wrong rule silently exposes data instead of failing loudly. Rules are read top to bottom and the first match wins, so ordering is part of the policy.

How authorization is evaluated. Transport, authentication, and authorization are independent: TLS protects the wire, the authenticator proves identity, and the access control plugin decides what that identity may do. File-based rules are evaluated top to bottom and the first match wins, so a broad `allow` rule above a narrow `deny` rule silently grants access. A row filter removes rows, while a column mask rewrites values, and both run inside the engine on every query.

```sql
SHOW GRANTS ON iceberg.analytics.orders;

SELECT customer_id, email, national_id
FROM iceberg.analytics.customers
LIMIT 5;
```

Common mistake: treating masking as encryption and assuming the raw value is unreachable. Masks are presentation rules evaluated at query time; a user with `SELECT` on the underlying table through another catalog or an ungoverned connector can still read the raw column. Combine masking with least-privilege grants, catalog-level rules, and encryption at rest. Test rules with a real non-admin account, because a wrong rule exposes data quietly.

## How do you write data with CTAS and INSERT?

Trino writes through connectors that implement the write SPI. `CREATE TABLE AS SELECT` and `INSERT` are the two primary tools.

```sql
CREATE TABLE iceberg.analytics.orders_daily
WITH (format = 'PARQUET', partitioning = ARRAY['order_date'])
AS
SELECT order_date, customer_id, count(*) AS orders, sum(amount) AS revenue
FROM iceberg.analytics.orders
GROUP BY order_date, customer_id;

INSERT INTO iceberg.analytics.orders_daily
SELECT order_date, customer_id, count(*), sum(amount)
FROM iceberg.analytics.orders
WHERE order_date = DATE '2026-10-04'
GROUP BY order_date, customer_id;
```

For tables that support it, row-level operations are available:

```sql
DELETE FROM iceberg.analytics.orders WHERE status = 'cancelled';

MERGE INTO iceberg.analytics.customers AS t
USING postgres.public.customers AS s ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET t.tier = s.tier, t.updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, tier, updated_at) VALUES (s.customer_id, s.tier, s.updated_at);
```

Control output file count with `task_max_writer_count`, `writer_scaling_min_data_processed`, and `scale_writers`. A CTAS that produces thousands of tiny files is the most common Trino write complaint, and compaction afterwards is the standard remedy.

How a write executes. `CREATE TABLE AS SELECT` and `INSERT` run a distributed write: each writer task receives rows and produces one or more files per partition. Connectors that support write partitioning route rows to writers by partition, and writer scaling adds writers only while the query is bottlenecked on writing. `task_max_writer_count` caps writers per task, `scale_writers` enables dynamic scaling, and `writer_scaling_min_data_processed` (120 MB default) sets how much uncompressed data a writer must process before another writer is added.

```sql
SET SESSION scale_writers = true;
SET SESSION task_max_writer_count = 8;
SET SESSION writer_scaling_min_data_processed = '256MB';

CREATE TABLE iceberg.analytics.orders_daily
WITH (format = 'PARQUET', partitioning = ARRAY['order_date'])
AS
SELECT order_date, customer_id, count(*) AS orders, sum(amount) AS revenue
FROM iceberg.analytics.orders
GROUP BY 1, 2;
```

Common mistake: writing thousands of tiny files and then blaming the reader. One writer per task and per partition means high parallelism produces many small files, which slows every later query. Target 128 MB to 512 MB files, enable writer scaling, and schedule compaction in the table format after the load.

## What connector-specific write limits should you know?

Not every connector is writable, and writable connectors differ in what they accept.

| Connector | Read | Insert | CTAS | Update/Delete/Merge | Notes |
|---|---|---|---|---|---|
| Iceberg | yes | yes | yes | yes | full table-format writes, partitioning in `WITH` |
| Delta Lake | yes | yes | yes | limited | let Spark or Databricks own heavy maintenance |
| Hive | yes | yes | yes | no | append-oriented; transactional tables need care |
| PostgreSQL | yes | yes | yes | yes | writes go to the source database |
| BigQuery | yes | yes | yes | yes | some versions reject `INSERT ... VALUES`; cost accrues on BigQuery |
| Kafka | yes | no | no | no | read-only, for inspection |
| Cassandra | yes | yes | yes | yes | the write path must respect the data model |

Other practical limits: some connectors reject `INSERT INTO ... VALUES` and require `INSERT INTO ... SELECT`, because a distributed engine has no efficient single-row path; type mapping differs per connector, especially for `varchar` and `timestamp with time zone`; and DDL such as `DROP COLUMN` may be unsupported or restricted when it would invalidate the underlying format.

How capabilities are exposed. Writes go through the connector write SPI. A connector advertises which statements it accepts through its metadata and procedures, so the same `INSERT` can succeed on Iceberg, be rejected on Kafka, and be expensive on PostgreSQL because it writes into the source database. Row-level `UPDATE`, `DELETE`, and `MERGE` are the narrowest capability; treat them as optional until you verify the connector.

```sql
SELECT catalog_name, schema_name, procedure_name
FROM system.metadata.table_procedures
WHERE catalog_name = 'iceberg'
ORDER BY schema_name, procedure_name;
```

Common mistake: assuming a connector that supports `INSERT` also supports `MERGE` or `DELETE`. Capabilities are per connector and sometimes per file format, so check the procedures and the connector documentation before designing a pipeline around row-level changes.

## How does Trino compare with Presto Spark SQL and Hive?

Trino and Presto share an ancestor but diverged. Getting the history and workload fit right is a common interview question.

| Engine | Model | Strength | Weakness |
|---|---|---|---|
| Trino | MPP, in-memory, no storage | interactive federated SQL, ANSI SQL, many connectors | not built for multi-TB shuffles without spilling |
| PrestoDB | MPP | compatible lineage, Meta-backed community | connector ecosystem and release cadence differ from Trino |
| Spark SQL | distributed batch and streaming | heavy ETL, ML, complex transformations, fault tolerance | higher latency to first result, JVM tuning overhead |
| Hive | batch on MapReduce or Tez | mature metastore, huge legacy install base | outdated execution model, poor interactive latency |

```text
Choosing in one line:
interactive and federated SQL across many sources -> Trino
large ETL transforms and ML feature builds       -> Spark SQL
legacy batch on an existing metastore             -> Hive, or migrate to Trino or Spark
```

Trino is not a scheduler and not a storage format. It reads Iceberg, Delta, Hive, Kafka, and databases; orchestration belongs to Airflow or Dagster, table semantics belong to Iceberg or Delta, and stream processing belongs to Flink. It is a poor fit for petabyte nightly shuffles, high-frequency single-row OLTP, or a tiny dataset with one source.

How the engines differ mechanically. Trino pipelines rows in memory between stages and ships a plan optimized for latency, which is why it returns first rows fast but struggles with a multi-terabyte shuffle. Spark SQL persists intermediate stages to disk, tolerates executor loss through lineage, and is built for long batch jobs. Hive runs on older execution engines and mainly survives as a metastore. The choice is about the shape of the workload, not about which engine is better.

```text
interactive, federated, many sources      -> Trino
petabyte nightly transforms, ML features  -> Spark SQL
legacy batch on an existing metastore     -> Hive, or migrate
stream processing, exactly-once sinks     -> Flink or Spark Structured Streaming
```

Common mistake: proposing Trino as a replacement for Spark for heavy ETL. Trino can run large queries, especially with fault-tolerant execution, but a multi-hour shuffle-heavy pipeline is still a batch-engine job. The strong answer combines them: Spark or Flink writes the tables, Trino serves fast SQL over them.

## How do you find the expensive stage and fix bad splitting?

Tuning starts by finding the stage that dominates wall time, not by changing a global property. `EXPLAIN (TYPE DISTRIBUTED)` prints one entry per fragment, and the fragment graph is the stage tree. The expensive stage is usually one that reads a huge table, one fed by a large `Exchange`, or one whose build tasks are skewed. `EXPLAIN ANALYZE` adds per-operator row counts, wall time, and peak memory, which is the fastest way to see where the time actually goes.

A scan stage creates one split per file or per byte range. `hive.max-split-size` caps split size, so a 2 GB Parquet file becomes about 16 splits at a 128 MB setting, and each task consumes `task_concurrency` splits at a time (16 by default). Small files hurt twice: 100,000 tiny files produce 100,000 splits, which the coordinator must enumerate and schedule, and each split carries fixed overhead that dwarfs the bytes it reads. Compact to files near 128 MB to 512 MB before tuning any property.

```sql
EXPLAIN (TYPE DISTRIBUTED)
SELECT o.order_date, count(*) AS orders
FROM iceberg.analytics.orders o
JOIN iceberg.analytics.customers c ON o.customer_id = c.customer_id
WHERE o.order_date >= DATE '2026-10-01'
GROUP BY 1;
```

Read the plan bottom-up: scan, then exchange, then aggregate. A large `RemoteSource` above a scan that should have been filtered means the predicate did not push down. Common mistake: raising `task_concurrency` or `max_hash_partition_count` first. No concurrency setting fixes a bad file layout.

In [ ]:
# Small files become splits; splits become tasks.
files = 100_000
avg_file_mb = 4
max_split_mb = 128
task_concurrency = 16

splits_per_file = (avg_file_mb + max_split_mb - 1) // max_split_mb  # ceil
splits = files * splits_per_file
tasks = -(-splits // task_concurrency)
compacted_files = (files * avg_file_mb) // 512

print(f"files={files} avg={avg_file_mb}MB -> splits={splits} tasks={tasks}")
print(f"after compaction to 512MB files -> {compacted_files} files")

assert splits == 100_000
assert tasks == 6250
assert compacted_files == 781

## How do you use statistics to choose join distribution and reduce shuffle?

The cost-based optimizer decides join order and join distribution from cardinality estimates. `optimizer.join-reordering-strategy=AUTOMATIC` costs candidate orders and falls back to `ELIMINATE_CROSS_JOINS` when statistics are missing; `optimizer.max-reordered-joins` (default 8) caps how many joins are reordered at once because the number of orders grows factorially. Session overrides are `join_reordering_strategy` and `max_reordered_joins`.

`join_distribution_type=AUTOMATIC` picks broadcast when the build side is under `join_max_broadcast_table_size` (100 MB default) and partitioned otherwise. Broadcast copies the build side to every worker, so a wrong broadcast multiplies network and memory by the worker count. A partitioned join shuffles both sides by the join key, which scales with the data rather than the cluster. Dynamic filtering then pushes values or ranges collected from the build side into the probe-side scan, pruning partitions and row groups before all splits are scheduled.

```sql
SET SESSION join_distribution_type = 'PARTITIONED';
SET SESSION join_reordering_strategy = 'AUTOMATIC';
SET SESSION max_hash_partition_count = 256;
SET SESSION task_concurrency = 16;

EXPLAIN (TYPE DISTRIBUTED)
SELECT f.order_id, d.region, sum(f.amount) AS revenue
FROM iceberg.analytics.fact_orders f
JOIN iceberg.analytics.dim_order d ON f.order_id = d.order_id
WHERE d.region = 'EMEA'
GROUP BY 1, 2;
```

Reduce shuffle before tuning it. Filter and project as early as possible, pre-aggregate before a join that fans out, and check `optimizer.push-aggregation-through-outer-join`. A partitioned join that moves 900 million rows of 120 bytes shuffles about 100 GB; the same query after a partial aggregate can move a few GB. Common mistake: forcing `BROADCAST` to hide a bad estimate instead of refreshing statistics.

## How do you spill safely and size resource groups?

Spill is a safety valve, not a speedup. When a query approaches `query.max-memory-per-node`, the memory manager revokes memory and an operator writes intermediate state to a local path. A hash join partitions its build table `task.concurrency` ways and spills a subset of partitions along with matching probe rows, so peak memory falls toward the largest partition rather than the whole build table.

Set `spill-enabled=true` and give `spiller-spill-path` a comma-separated list of local NVMe mount points. Never spill to the drive that holds the JVM logs, and never to network storage. `max-spill-per-node` caps all queries on a node and `query-max-spill-per-node` caps one query on one node, both defaulting to 100 GB. `spill-compression-codec` accepts `LZ4` or `ZSTD` and trades CPU for disk IO. Trino treats the paths as independent disks, so plain JBOD beats RAID here.

```text
etc/config.properties
spill-enabled=true
spiller-spill-path=/mnt/nvme0/trino-spill,/mnt/nvme1/trino-spill
max-spill-per-node=400GB
query-max-spill-per-node=100GB
spill-compression-codec=ZSTD
```

Resource groups queue instead of failing. A query waits when its group exceeds `softMemoryLimit` or `hardConcurrencyLimit`, and is rejected when `maxQueued` is full. Give ETL a high memory share and modest concurrency, give ad-hoc the opposite, use `schedulingWeight` to bias `weighted_fair`, and set `jmxExport` so queue depth is visible. Common mistake: spilling to a network mount or the system drive, which turns a slow query into a failed cluster.

In [ ]:
# ILLUSTRATIVE - requires trino: install to run, or read on.
# This cell needs the trino client and a reachable cluster; review only.
from trino.dbapi import connect

cur = connect(host="trino.acme.internal", port=8080, user="bi_service",
              catalog="iceberg", schema="analytics").cursor()
cur.execute("SET SESSION join_distribution_type = 'PARTITIONED'")
cur.execute("SET SESSION spill_enabled = true")
cur.execute(
    "EXPLAIN (TYPE DISTRIBUTED) "
    "SELECT customer_id, count(*) FROM orders GROUP BY customer_id"
)
for row in cur.fetchall():
    print(row[0])

## What are the connector-specific pushdown limits?

Pushdown is a negotiation between the optimizer and the connector. The optimizer produces a `Domain` per column; the connector accepts the subset of predicate shapes it can translate and ignores the rest. Rows that survive the connector filter are re-checked by Trino, so a missed pushdown is a performance problem, never a correctness problem. The IO plan shows the accepted domains, and an unbounded domain means the connector could not help.

| Connector | Predicate pushdown | File or partition pruning | Notes |
|---|---|---|---|
| Iceberg | yes | manifests, column stats, `$partitions`, `$files` | hidden partitioning keeps filters on the base column |
| Delta Lake | yes | `_delta_log`, partition and column stats | open format, often written by Spark |
| Hive | yes | metastore partitions, ORC or Parquet row groups | depends on a healthy metastore |
| PostgreSQL | generated SQL expressions | source indexes | remote scan cost still applies |
| BigQuery | API filters | API pruning | bytes billed still accrue on BigQuery |
| Kafka | none | none | read-only, full decode of matching splits |

```sql
EXPLAIN (TYPE IO)
SELECT count(*) FROM postgres.public.customers
WHERE upper(email) = 'A@B.COM';
```

Common mistake: writing a predicate the connector cannot translate. A function on the column, an implicit cast, or a `varchar` literal compared with an integer turns into a full scan. Keep predicates as bare comparisons on typed literals, and check the IO plan before blaming the connector.

## Trino exercises

These exercises use only the Python standard library and model Trino planning and sizing arithmetic with small, runnable calculations. Each one states the task, gives a hint, reports the expected result, and ends with a code cell that prints and asserts the answer. Run them with `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are illustrative. The formulas are the part that matters: splits come from file sizes, tasks from splits and `task_concurrency`, broadcast from build-side bytes, spill peak from build bytes divided by workers and `task_concurrency`, and data movement from rows times row width. Treat the asserted values as a check on the arithmetic, not as a prediction of a real cluster.

### Try it 1. Decide broadcast versus partitioned join

**Task.** Given three tables, compute each join candidate's build-side size in bytes and classify it as `BROADCAST` when it fits under `join_max_broadcast_table_size` (100 MB) and `PARTITIONED` otherwise.

**Hint.** `bytes = rows * row_bytes`; compare with `100 * 1024 ** 2`.

**Expected result.** `{'dim_date': 'BROADCAST', 'dim_customer': 'PARTITIONED'}` with sizes about 4.6 MiB and 305.2 MiB.

In [ ]:
tables = {
    "dim_date": {"rows": 120_000, "row_bytes": 40},
    "dim_customer": {"rows": 4_000_000, "row_bytes": 80},
}
MAX_BROADCAST = 100 * 1024 ** 2


def distribution(table):
    size = table["rows"] * table["row_bytes"]
    return "BROADCAST" if size <= MAX_BROADCAST else "PARTITIONED"


plan = {name: distribution(t) for name, t in tables.items()}
for name, t in tables.items():
    mib = t["rows"] * t["row_bytes"] / 1024 ** 2
    print(f"{name:14s} {mib:8.1f} MiB -> {plan[name]}")

assert plan == {"dim_date": "BROADCAST", "dim_customer": "PARTITIONED"}

### Try it 2. Compute split count from file count and size

**Task.** A table holds 9,000 files of 8 MB, 2,500 files of 40 MB, and 500 files of 300 MB. With a 128 MB maximum split size, count the splits and the tasks at `task_concurrency = 16`.

**Hint.** `splits = sum(ceil(file_size / max_split_size))` and `tasks = ceil(splits / task_concurrency)`.

**Expected result.** `13000` splits, `813` tasks, and about `314.45` GiB of data.

In [ ]:
import math

MB = 1024 ** 2
files = [8 * MB] * 9000 + [40 * MB] * 2500 + [300 * MB] * 500
max_split_size = 128 * MB
task_concurrency = 16

splits = sum(math.ceil(size / max_split_size) for size in files)
tasks = math.ceil(splits / task_concurrency)
gib = sum(files) / 1024 ** 3

print(f"splits = {splits}")
print(f"tasks  = {tasks}")
print(f"data   = {gib:.2f} GiB")

assert splits == 13000
assert tasks == 813
assert round(gib, 2) == 314.45

### Try it 3. Pick a partition and file layout from query patterns

**Task.** Choose a transform per column: a time transform for a range filter, `identity` for a low-cardinality equality filter, and `bucket(64, ...)` for a high-cardinality equality filter. Then size a 4 TB table at a 512 MB target file size across 365 daily partitions.

**Hint.** `files = ceil(total_bytes / target_bytes)` and `files_per_partition = ceil(files / partitions)`.

**Expected result.** `days(order_ts)`, `bucket(64, customer_id)`, `identity(region)`; `8192` files total and `23` per partition.

In [ ]:
import math

patterns = {
    "order_ts": ("range", 365),
    "customer_id": ("equality", 2_000_000),
    "region": ("equality", 12),
}


def choose_transform(column, kind, cardinality):
    if kind == "range":
        return f"days({column})"
    if cardinality <= 1000:
        return f"identity({column})"
    return f"bucket(64, {column})"


transforms = {c: choose_transform(c, k, n) for c, (k, n) in patterns.items()}
for column, transform in transforms.items():
    print(f"{column:12s} -> {transform}")

total_bytes = 4 * 1024 ** 4
target_bytes = 512 * 1024 ** 2
partitions = 365
files = math.ceil(total_bytes / target_bytes)
files_per_partition = math.ceil(files / partitions)
print(f"files = {files}, per partition = {files_per_partition}")

assert transforms["order_ts"] == "days(order_ts)"
assert transforms["customer_id"] == "bucket(64, customer_id)"
assert transforms["region"] == "identity(region)"
assert files == 8192 and files_per_partition == 23

### Try it 4. Estimate spill peak from operator memory

**Task.** A hash join build side is 300 GB on a 40-worker cluster. Compute the peak memory per task with `task_concurrency = 16` and with `task_concurrency = 4`, and report whether each fits a per-node query limit of 8 GB.

**Hint.** `peak_mib = build_gb * 1024 / workers / task_concurrency`; the join spills down toward one partition of the build table.

**Expected result.** `480.0` MiB at concurrency 16 (fits) and `1920.0` MiB at concurrency 4 (fits); both under the 8192 MiB limit.

In [ ]:
build_gb = 300
workers = 40
node_query_limit_mib = 8 * 1024


def peak_mib(task_concurrency):
    return build_gb * 1024 / workers / task_concurrency


for concurrency in (16, 4):
    peak = peak_mib(concurrency)
    verdict = "fits" if peak <= node_query_limit_mib else "spill or fail"
    print(f"task_concurrency={concurrency:2d} -> peak {peak:7.1f} MiB -> {verdict}")

assert peak_mib(16) == 480.0
assert peak_mib(4) == 1920.0
assert peak_mib(4) < node_query_limit_mib

### Try it 5. Choose resource group limits from concurrency

**Task.** A cluster has 40 workers with 64 GB each, so 2,560 GB of query memory. ETL gets a 60 percent memory share with roughly 120 GB per query; ad-hoc gets 30 percent with roughly 15 GB per query. Compute each group's `hardConcurrencyLimit` and report the shares.

**Hint.** `concurrency = floor(cluster_memory * share / memory_per_query)`.

**Expected result.** ETL concurrency `12`, ad-hoc concurrency `51`, shares `60` and `30` percent.

In [ ]:
cluster_gb = 40 * 64
groups = {
    "etl": {"share": 0.60, "memory_per_query_gb": 120},
    "adhoc": {"share": 0.30, "memory_per_query_gb": 15},
}

limits = {}
for name, g in groups.items():
    concurrency = int(cluster_gb * g["share"] // g["memory_per_query_gb"])
    limits[name] = concurrency
    print(f"{name:6s} share={g['share']:.0%} -> hardConcurrencyLimit={concurrency}")

print("cluster memory GB =", cluster_gb)
assert limits == {"etl": 12, "adhoc": 51}

### Try it 6. Compare pushdown with pull for a filter

**Task.** A source table has 200 million rows at 200 bytes each, and a filter keeps 0.5 percent of them. Compare the bytes transferred when Trino pulls every row and filters locally with the bytes transferred when the connector filters before sending.

**Hint.** `transferred = rows * row_bytes * selectivity` for pushdown, and `rows * row_bytes` for pull.

**Expected result.** `40.0` GB pulled, `0.2` GB pushed, a `200.0` times reduction in network traffic.

In [ ]:
rows = 200_000_000
row_bytes = 200
selectivity = 0.005

pull_gb = rows * row_bytes / 10 ** 9
push_gb = pull_gb * selectivity
reduction = pull_gb / push_gb

print(f"pull = {pull_gb:.1f} GB, push = {push_gb:.1f} GB, reduction = {reduction:.1f}x")
assert round(pull_gb, 1) == 40.0
assert round(push_gb, 1) == 0.2
assert round(reduction, 1) == 200.0

### Try it 7. Compute a stage's data movement from row width

**Task.** A repartition exchange moves 1.2 billion rows of 84 bytes. Compute the bytes moved in GiB. Then a partial aggregate shrinks the input to 60 million rows of 64 bytes before the exchange: compute the new size and the reduction.

**Hint.** `bytes = rows * row_bytes` and `gib = bytes / 1024 ** 3`.

**Expected result.** `93.9` GiB before and `3.6` GiB after, a reduction of about `26.25` times.

In [ ]:
rows_before, width_before = 1_200_000_000, 84
rows_after, width_after = 60_000_000, 64

gib_before = rows_before * width_before / 1024 ** 3
gib_after = rows_after * width_after / 1024 ** 3
reduction = gib_before / gib_after

print(f"before = {gib_before:.1f} GiB")
print(f"after  = {gib_after:.1f} GiB")
print(f"reduction = {reduction:.2f}x")

assert round(gib_before, 1) == 93.9
assert round(gib_after, 1) == 3.6
assert round(reduction, 2) == 26.25

### Try it 8. Estimate the win from dynamic filtering

**Task.** A fact table has 5,000 partitions of 4 GB each. A filtered date dimension selects 12 keys, and each key matches one fact partition. Compute how many partitions are scanned with and without dynamic filtering, and the bytes saved.

**Hint.** Without dynamic filtering every partition is scanned; with it, `scanned = keys`; `saved_gb = (total - scanned) * partition_gb`.

**Expected result.** `12` partitions scanned, `4988` pruned, and `19952` GB saved (about 99.8 percent).

In [ ]:
total_partitions = 5000
partition_gb = 4
dimension_keys = 12

scanned = dimension_keys
pruned = total_partitions - scanned
saved_gb = pruned * partition_gb
saved_fraction = saved_gb / (total_partitions * partition_gb)

print(f"scanned partitions = {scanned}")
print(f"pruned partitions  = {pruned}")
print(f"bytes saved        = {saved_gb} GB ({saved_fraction:.1%})")

assert scanned == 12
assert pruned == 4988
assert saved_gb == 19952
assert round(saved_fraction, 3) == 0.998

## Real world use cases for Trino and Presto

### 1. Federated analytics across the lake and operational databases

A retail analytics team keeps orders in Iceberg on S3, customers in PostgreSQL, and marketing data in BigQuery. Analysts run one statement instead of exporting three extracts nightly.

```sql
SELECT c.tier,
       date_trunc('month', o.order_ts) AS month,
       sum(o.amount) AS revenue
FROM iceberg.analytics.orders o
JOIN postgres.public.customers c ON o.customer_id = c.customer_id
JOIN bigquery.marketing.campaigns m ON m.customer_id = c.customer_id
WHERE o.order_ts >= TIMESTAMP '2026-01-01 00:00:00'
GROUP BY 1, 2
ORDER BY 2, 1;
```

Why Trino: connectors replace nightly exports and the join happens in the engine. Selective predicates keep each source's scan small, and the warehouse never has to hold a copy of operational data.

### 2. Lakehouse SQL while Spark owns the writes

Spark and dbt write Iceberg tables; Trino serves the interactive layer over the same files and snapshots.

In [ ]:
# ILLUSTRATIVE - requires trino: install to run, or read on.
from trino.dbapi import connect

cur = connect(
    host="trino.acme.internal", port=8080, user="bi_service",
    catalog="iceberg", schema="analytics",
).cursor()
cur.execute(
    "SELECT order_date, sum(amount) AS revenue FROM orders "
    "WHERE order_date BETWEEN DATE '2026-10-01' AND DATE '2026-10-04' "
    "GROUP BY order_date ORDER BY order_date"
)
for order_date, revenue in cur.fetchall():
    print(order_date, revenue)

Why Trino: fast planning over Iceberg metadata with no data duplication. Spark keeps its strengths for large transformations while Trino answers questions.

### 3. Migrating off a monolithic warehouse with a federation bridge

A company moves from a legacy warehouse to a lakehouse over several quarters. During the transition, Trino joins tables that still live in the old system with tables already migrated.

```sql
SELECT l.customer_id, l.lifetime_value, w.last_invoice_date
FROM iceberg.analytics.customer_value l
LEFT JOIN postgres_legacy.dw.invoices w ON l.customer_id = w.customer_id
WHERE l.snapshot_date = DATE '2026-10-04';
```

Why Trino: a federation bridge removes the need for a big-bang cutover. Each migrated table is one less dependency on the legacy system, and the SQL above never changes.

### 4. Ad-hoc log and event investigation

A platform team answers incident questions across Kafka, S3 Parquet, and a metrics database without building an ETL job per question.

```sql
SELECT src, count(*) AS errors
FROM (
  SELECT 'kafka' AS src FROM kafka.acme."app-logs"
  WHERE level = 'ERROR' AND _timestamp > now() - INTERVAL '30' MINUTE
  UNION ALL
  SELECT 'archive' FROM hive.warehouse.app_logs
  WHERE level = 'ERROR' AND log_date = current_date
) t
GROUP BY src;
```

Why Trino: the Kafka connector's timestamp and offset columns make producer debugging possible, and a UNION across file-based sources gives a complete picture in one query.

### 5. Governed self-service BI with row filters and masking

Support and finance share tables but must see different rows and masked columns. Policy is enforced in Trino rather than duplicated in every BI tool.

```json
{
  "tables": [
    { "group": "finance", "catalog": "iceberg", "schema": "analytics", "table": "orders", "privileges": ["SELECT"] },
    { "group": "support", "catalog": "iceberg", "schema": "analytics", "table": "orders",
      "privileges": ["SELECT"], "filter": "region = 'EMEA'",
      "columns": [
        { "name": "customer_email", "mask": "'***@***'" }
      ] }
  ]
}
```

```sql
SELECT order_id, region, customer_email FROM iceberg.analytics.orders LIMIT 10;
```

Why Trino: one policy layer applies to the CLI, notebooks, and BI tools alike, which is far easier to audit than per-tool row filters and duplicated masking views.

A strong answer for any of these cases names three things: the source of each table and its connector, what moves across the network and where the join happens, and how the layout or statistics keep the scan small. Common mistake: describing Trino as a warehouse. It has no storage, so every performance story is a story about connectors, splits, and the exchange.